<a href="https://colab.research.google.com/github/Shahul187/aml-alert-triage/blob/main/notebooks/04_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

base = '/content/drive/MyDrive/aml-project/'
train_f = pd.read_parquet(base + 'train_features.parquet')
test_f  = pd.read_parquet(base + 'test_features.parquet')
FEATURES = pd.read_csv(base + 'feature_list.csv')['feature'].tolist()

X_train, y_train = train_f[FEATURES], train_f['Is_laundering']
X_test,  y_test  = test_f[FEATURES],  test_f['Is_laundering']

print(f"train {X_train.shape} | {y_train.sum():,} criminal ({y_train.mean()*100:.3f}%)")
print(f"test  {X_test.shape} | {y_test.sum():,} criminal ({y_test.mean()*100:.3f}%)")
print(f"\nFeatures: {FEATURES}")

Mounted at /content/drive
train (163419, 25) | 3,355 criminal (2.053%)
test  (56544, 25) | 1,416 criminal (2.504%)

Features: ['Amount', 'alert_r1', 'alert_r2', 'alert_r3', 'alert_r4', 'hist_n', 'hist_mean', 'hist_std', 'hist_max', 'hist_recv_ratio', 'has_history', 'amt_vs_mean', 'amt_vs_max', 'amt_zscore', 'r_n', 'r_mean', 'r_n_senders', 'r_sender_ratio', 'has_recv_history', 'recv_is_passthrough', 'sender_is_passthrough', 'pair_seen', 'is_cash', 'is_crossborder', 'same_currency']


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

logreg = Pipeline([
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

logreg.fit(X_train, y_train)
p_lr = logreg.predict_proba(X_test)[:, 1]

print("Trained. Score distribution:")
print(pd.Series(p_lr).describe().round(4))

Trained. Score distribution:
count    56544.0000
mean         0.1408
std          0.2924
min          0.0000
25%          0.0000
50%          0.0003
75%          0.0843
max          1.0000
dtype: float64


In [3]:
from sklearn.metrics import roc_auc_score, average_precision_score

def evaluate(y, scores, name):
    auc = roc_auc_score(y, scores)
    ap = average_precision_score(y, scores)
    base = y.mean()
    print(f"--- {name} ---")
    print(f"  ROC-AUC:      {auc:.4f}")
    print(f"  PR-AUC:       {ap:.4f}  (baseline {base:.4f}, lift {ap/base:.1f}x)")
    return {'model': name, 'roc_auc': auc, 'pr_auc': ap}

scores = []
scores.append(evaluate(y_test, p_lr, "Logistic regression"))

--- Logistic regression ---
  ROC-AUC:      0.6149
  PR-AUC:       0.0483  (baseline 0.0250, lift 1.9x)


In [4]:
skewed = ['Amount','hist_mean','hist_std','hist_max','r_mean',
          'amt_vs_mean','amt_vs_max','pair_seen','hist_n','r_n']

X_train_lg = X_train.copy()
X_test_lg  = X_test.copy()
for c in skewed:
    X_train_lg[c] = np.log1p(X_train_lg[c].clip(lower=0))
    X_test_lg[c]  = np.log1p(X_test_lg[c].clip(lower=0))

logreg2 = Pipeline([
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])
logreg2.fit(X_train_lg, y_train)
p_lr2 = logreg2.predict_proba(X_test_lg)[:, 1]

scores.append(evaluate(y_test, p_lr2, "Logistic regression (log)"))

--- Logistic regression (log) ---
  ROC-AUC:      0.5240
  PR-AUC:       0.0257  (baseline 0.0250, lift 1.0x)


In [5]:
from xgboost import XGBClassifier

ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {ratio:.1f}")

xgb = XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=ratio, eval_metric='aucpr',
    random_state=42, n_jobs=-1
)
xgb.fit(X_train, y_train)
p_xgb = xgb.predict_proba(X_test)[:, 1]

scores.append(evaluate(y_test, p_xgb, "XGBoost"))

scale_pos_weight: 47.7
--- XGBoost ---
  ROC-AUC:      0.7957
  PR-AUC:       0.5112  (baseline 0.0250, lift 20.4x)


In [6]:
imp = pd.DataFrame({
    'feature': FEATURES,
    'gain': xgb.feature_importances_
}).sort_values('gain', ascending=False)

print(imp.to_string(index=False))

              feature     gain
            pair_seen 0.360430
       r_sender_ratio 0.130489
              is_cash 0.081038
             alert_r4 0.064190
       is_crossborder 0.046626
  recv_is_passthrough 0.042178
             alert_r3 0.033647
                  r_n 0.024443
          r_n_senders 0.022736
               hist_n 0.022638
               r_mean 0.021642
      hist_recv_ratio 0.021568
             alert_r1 0.018970
             hist_std 0.017996
           amt_vs_max 0.016411
sender_is_passthrough 0.015611
               Amount 0.010116
             hist_max 0.009789
            hist_mean 0.009321
             alert_r2 0.009079
          amt_vs_mean 0.008602
        same_currency 0.006964
           amt_zscore 0.005518
          has_history 0.000000
     has_recv_history 0.000000


In [7]:
suspect = ['pair_seen','hist_max','hist_mean','hist_std','hist_n','r_n','r_mean']
safe = [f for f in FEATURES if f not in suspect]
print(f"Retraining on {len(safe)} features (dropped {len(suspect)})")

xgb_safe = XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=ratio, eval_metric='aucpr',
    random_state=42, n_jobs=-1
)
xgb_safe.fit(X_train[safe], y_train)
p_safe = xgb_safe.predict_proba(X_test[safe])[:, 1]

evaluate(y_test, p_safe, "XGBoost (no account-count features)")

Retraining on 18 features (dropped 7)
--- XGBoost (no account-count features) ---
  ROC-AUC:      0.7582
  PR-AUC:       0.3184  (baseline 0.0250, lift 12.7x)


{'model': 'XGBoost (no account-count features)',
 'roc_auc': np.float64(0.7582274690253479),
 'pr_auc': np.float64(0.3184163878080314)}

In [8]:
DROP = ['pair_seen', 'has_history', 'has_recv_history']
FEATURES_V2 = [f for f in FEATURES if f not in DROP]
print(f"{len(FEATURES_V2)} features (dropped {DROP})")

xgb2 = XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=ratio, eval_metric='aucpr',
    random_state=42, n_jobs=-1
)
xgb2.fit(X_train[FEATURES_V2], y_train)
p2 = xgb2.predict_proba(X_test[FEATURES_V2])[:, 1]

scores.append(evaluate(y_test, p2, "XGBoost v2 (no pair_seen)"))

22 features (dropped ['pair_seen', 'has_history', 'has_recv_history'])
--- XGBoost v2 (no pair_seen) ---
  ROC-AUC:      0.7786
  PR-AUC:       0.5042  (baseline 0.0250, lift 20.1x)


In [9]:
imp2 = pd.DataFrame({'feature': FEATURES_V2, 'gain': xgb2.feature_importances_}
                    ).sort_values('gain', ascending=False)
print(imp2.head(10).to_string(index=False))

print(f"\nTop-1 share: {imp2['gain'].iloc[0]:.3f}")
print(f"Top-3 share: {imp2['gain'].head(3).sum():.3f}")

            feature     gain
     r_sender_ratio 0.265900
            is_cash 0.150413
     is_crossborder 0.067957
           alert_r4 0.052601
recv_is_passthrough 0.049984
    hist_recv_ratio 0.040038
           alert_r3 0.036268
        r_n_senders 0.035093
                r_n 0.034672
             r_mean 0.034630

Top-1 share: 0.266
Top-3 share: 0.484


In [10]:
acct = ['hist_n','hist_mean','hist_std','hist_max','hist_recv_ratio','r_n','r_mean']
lean = [f for f in FEATURES_V2 if f not in acct]
print(f"Lean model: {len(lean)} features")

xgb_lean = XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=ratio, eval_metric='aucpr',
    random_state=42, n_jobs=-1
)
xgb_lean.fit(X_train[lean], y_train)
p_lean = xgb_lean.predict_proba(X_test[lean])[:, 1]
evaluate(y_test, p_lean, "XGBoost lean (no account aggregates)")

Lean model: 15 features
--- XGBoost lean (no account aggregates) ---
  ROC-AUC:      0.7157
  PR-AUC:       0.2765  (baseline 0.0250, lift 11.0x)


{'model': 'XGBoost lean (no account aggregates)',
 'roc_auc': np.float64(0.7156638079883121),
 'pr_auc': np.float64(0.27651461379328623)}

In [11]:
import joblib
joblib.dump(xgb2, base + 'xgb_model.pkl')
np.save(base + 'test_scores.npy', p2)
pd.DataFrame(scores).to_csv(base + 'model_comparison.csv', index=False)
imp2.to_csv(base + 'feature_importance.csv', index=False)
print("Saved.")

Saved.


In [12]:
known = test_f['hist_n'] > 0
print(f"Known accounts:   {known.sum():,} rows | {y_test[known].sum():,} criminal ({y_test[known].mean()*100:.2f}%)")
print(f"Unseen accounts:  {(~known).sum():,} rows | {y_test[~known].sum():,} criminal ({y_test[~known].mean()*100:.2f}%)")

print()
evaluate(y_test[known], p2[known.values], "v2 on KNOWN accounts")
print()
evaluate(y_test[~known], p2[(~known).values], "v2 on UNSEEN accounts")

Known accounts:   38,597 rows | 1,078 criminal (2.79%)
Unseen accounts:  17,947 rows | 338 criminal (1.88%)

--- v2 on KNOWN accounts ---
  ROC-AUC:      0.8538
  PR-AUC:       0.5887  (baseline 0.0279, lift 21.1x)

--- v2 on UNSEEN accounts ---
  ROC-AUC:      0.5213
  PR-AUC:       0.2582  (baseline 0.0188, lift 13.7x)


{'model': 'v2 on UNSEEN accounts',
 'roc_auc': np.float64(0.5212542604457578),
 'pr_auc': np.float64(0.2582329871155826)}

In [13]:
def precision_at_k(y, s, ks=(0.05,0.10,0.20,0.30,0.50)):
    order = np.argsort(-s)
    y_sorted = np.asarray(y)[order]
    total_crime = y_sorted.sum()
    rows = []
    for k in ks:
        n = int(len(y_sorted)*k)
        caught = y_sorted[:n].sum()
        rows.append({
            'review_top_%': f"{k*100:.0f}%",
            'alerts_reviewed': n,
            'caught': caught,
            'precision_%': round(caught/n*100, 2),
            'recall_%': round(caught/total_crime*100, 1),
            'workload_saved_%': round((1-k)*100)
        })
    return pd.DataFrame(rows)

pk = precision_at_k(y_test, p2)
print(pk.to_string(index=False))

review_top_%  alerts_reviewed  caught  precision_%  recall_%  workload_saved_%
          5%             2827     729        25.79      51.5                95
         10%             5654     770        13.62      54.4                90
         20%            11308     852         7.53      60.2                80
         30%            16963     964         5.68      68.1                70
         50%            28272    1141         4.04      80.6                50


In [14]:
for label, mask in [('KNOWN accounts', known.values), ('UNSEEN accounts', (~known).values)]:
    print(f"\n=== {label} ===")
    print(precision_at_k(y_test[mask], p2[mask]).to_string(index=False))


=== KNOWN accounts ===
review_top_%  alerts_reviewed  caught  precision_%  recall_%  workload_saved_%
          5%             1929     631        32.71      58.5                95
         10%             3859     673        17.44      62.4                90
         20%             7719     763         9.88      70.8                80
         30%            11579     843         7.28      78.2                70
         50%            19298     965         5.00      89.5                50

=== UNSEEN accounts ===
review_top_%  alerts_reviewed  caught  precision_%  recall_%  workload_saved_%
          5%              897      94        10.48      27.8                95
         10%             1794     100         5.57      29.6                90
         20%             3589     115         3.20      34.0                80
         30%             5384     141         2.62      41.7                70
         50%             8973     171         1.91      50.6                50


In [15]:
pk.to_csv(base + 'precision_at_k.csv', index=False)
seg = pd.DataFrame([
    {'segment':'known','n':int(known.sum()),'roc_auc':0.8538,'pr_auc':0.5887,'lift':21.1},
    {'segment':'unseen','n':int((~known).sum()),'roc_auc':0.5213,'pr_auc':0.2582,'lift':13.7},
])
seg.to_csv(base + 'segment_performance.csv', index=False)
print("Saved.")

Saved.
